# Day 27 (Extension): ViT for Image Restoration — CNN vs ViT vs Hybrid

> **Goal:** Adapt the Vision Transformer (ViT) for image restoration tasks. Build a pure ViT restoration model, a hybrid CNN+ViT model, and compare all three approaches (CNN U-Net vs ViT vs Hybrid) on denoising and deblurring.

> **Prerequisite:** Day 27 (CNN vs ViT — Head-to-Head Comparison)

---

## Why This Extension Exists

You've seen that ViT can classify images. But **can ViT restore images?** Yes — same backbone, different head.

| Component | Classifier ViT (Day 25) | Restoration ViT (Today) |
|---|---|---|
| **Head** | [CLS] token → MLP → class logits | Unpatchify + Conv2d → output image |
| **Loss** | CrossEntropy (label smoothing) | MSE (pixel comparison) |
| **Output** | 10 class probabilities | Restored image (3×32×32) |

### When Does Global Attention Help Restoration?

- **Periodic textures** (fabric, grass, bricks) → CNN does well locally
- **Large missing regions or structured noise** → ViT uses global context
- **Hybrid model** = CNN for local features + ViT for global context + CNN decoder for reconstruction → best of both worlds

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import math
import time
from scipy.ndimage import gaussian_filter

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

---
## 1. Degradation Pipeline (Reused from Notebook 2)

Same synthetic degradation functions from Notebook 2 — ensures fair comparison.

In [ ]:
def add_gaussian_noise(img, sigma=0.1):
    noise = torch.randn_like(img) * sigma
    return torch.clamp(img + noise, 0, 1)

def apply_gaussian_blur(img, kernel_size=5, sigma=1.5):
    img_np = img.cpu().numpy()
    blurred = np.zeros_like(img_np)
    for c in range(img_np.shape[0]):
        blurred[c] = gaussian_filter(img_np[c], sigma=sigma, truncate=(kernel_size-1)/(2*sigma))
    return torch.from_numpy(blurred).to(img.device)

class DegradedCIFAR10(Dataset):
    def __init__(self, base_dataset, degradation_type="noise", sigma=0.1, blur_sigma=1.5, blur_kernel=5):
        self.base = base_dataset
        self.degradation_type = degradation_type
        self.sigma = sigma
        self.blur_sigma = blur_sigma
        self.blur_kernel = blur_kernel

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, _ = self.base[idx]
        CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
        CIFAR_STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
        img_01 = img * CIFAR_STD + CIFAR_MEAN
        img_01 = torch.clamp(img_01, 0, 1)

        if self.degradation_type == "noise":
            degraded = add_gaussian_noise(img_01, sigma=self.sigma)
        elif self.degradation_type == "blur":
            degraded = apply_gaussian_blur(img_01, sigma=self.blur_sigma, kernel_size=self.blur_kernel)
        else:
            raise ValueError(f"Unknown degradation: {self.degradation_type}")

        degraded_norm = (degraded - CIFAR_MEAN.to(degraded.device)) / CIFAR_STD.to(degraded.device)
        return degraded_norm, img

---
## 2. Metrics (Reused from Notebook 2)

PSNR and SSIM — implemented from scratch as before.

In [ ]:
@torch.no_grad()
def compute_psnr(img_pred, img_true, max_val=1.0):
    mse = F.mse_loss(img_pred, img_true)
    if mse == 0:
        return float('inf')
    return 10 * torch.log10(max_val**2 / mse).item()

@torch.no_grad()
def compute_ssim(img_pred, img_true, window_size=11, C1=0.01, C2=0.03):
    gauss = torch.tensor([np.exp(-0.5 * ((x - window_size//2) ** 2) / (1.5 ** 2))
                          for x in range(window_size)], dtype=torch.float32)
    gauss = gauss / gauss.sum()
    window_2d = gauss[:, None] * gauss[None, :]
    window_2d = window_2d.expand(img_pred.shape[1], 1, window_size, window_size).to(img_pred.device)
    def gf(x): return F.conv2d(x, window_2d, padding=window_size//2, groups=x.shape[1])
    mu_x, mu_y = gf(img_pred), gf(img_true)
    sx2 = gf(img_pred**2) - mu_x**2
    sy2 = gf(img_true**2) - mu_y**2
    sxy = gf(img_pred * img_true) - mu_x * mu_y
    C1, C2 = C1**2, C2**2
    ssim_map = ((2*mu_x*mu_y + C1) * (2*sxy + C2)) / ((mu_x**2 + mu_y**2 + C1) * (sx2 + sy2 + C2))
    return ssim_map.mean().item()

print("✅ Metrics loaded")

---
## 3. Data Setup

We'll use the same CIFAR-10 split as Day 25 (ViT day), with synthetic denoising and deblurring datasets.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

full_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
train_data, val_data = random_split(full_train, [45000, 5000], generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 64

# Denoising datasets (σ=0.2)
train_dn = DegradedCIFAR10(train_data, "noise", sigma=0.2)
val_dn   = DegradedCIFAR10(val_data,   "noise", sigma=0.2)
test_dn  = DegradedCIFAR10(test_data,  "noise", sigma=0.2)

train_loader_dn = DataLoader(train_dn, batch_size=BATCH_SIZE, shuffle=True)
val_loader_dn   = DataLoader(val_dn,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_dn  = DataLoader(test_dn,  batch_size=BATCH_SIZE, shuffle=False)

# Deblurring datasets (PSF σ=1.5, kernel=5)
train_bl = DegradedCIFAR10(train_data, "blur", blur_sigma=1.5, blur_kernel=5)
val_bl   = DegradedCIFAR10(val_data,   "blur", blur_sigma=1.5, blur_kernel=5)
test_bl  = DegradedCIFAR10(test_data,  "blur", blur_sigma=1.5, blur_kernel=5)

train_loader_bl = DataLoader(train_bl, batch_size=BATCH_SIZE, shuffle=True)
val_loader_bl   = DataLoader(val_bl,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_bl  = DataLoader(test_bl,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Datasets created — Train: {len(train_dn)}, Val: {len(val_dn)}, Test: {len(test_dn)}")

---
## 4. ViT Building Blocks (Recap from Day 25)

These are the same components we built on Day 25: PatchEmbedding, MultiHeadAttention, FeedForward, TransformerBlock. 

The key difference from the classifier ViT: we add an **Unpatchify** decoder that converts patch tokens back to an image.

In [ ]:
class PatchEmbedding(nn.Module):
    """Image → patch sequence with positional encoding."""
    def __init__(self, img_size=32, patch_size=4, in_channels=3, d_model=256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_size = patch_size
        self.patch_proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, d_model) * 0.02)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_proj(x).flatten(2).transpose(1, 2)  # (B, N, d_model)
        x = x + self.pos_embed
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x, return_weights=False):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.d_k).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv.unbind(0)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_drop(attn)
        out = (attn @ V).transpose(1, 2).reshape(B, N, C)
        out = self.proj_drop(self.proj(out))
        if return_weights:
            return out, attn
        return out

class FeedForward(nn.Module):
    def __init__(self, d_model, expansion=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * expansion),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * expansion, d_model),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, expansion=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = FeedForward(d_model, expansion, dropout)

    def forward(self, x, return_weights=False):
        if return_weights:
            attn_out, weights = self.attn(self.norm1(x), return_weights=True)
            x = x + attn_out
        else:
            x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        if return_weights:
            return x, weights
        return x

print("✅ ViT building blocks defined!")

### Unpatchify Decoder

For restoration, we need to convert patch tokens back to an image. This is the reverse of patch embedding:

- **Patch embed:** Image (B, 3, 32, 32) → patches (B, N, d_model)
- **Transformer encoder:** Self-attention across patches
- **Unpatchify:** (B, N, d_model) → reshape → Conv2d → (B, 3, 32, 32)

Unpatchify is essentially: reshape the sequence back to a grid, then apply a convolution to smooth out patch boundaries.

In [ ]:
class Unpatchify(nn.Module):
    """Convert patch sequence back to image.
    
    Input: (B, num_patches, d_model)
    Output: (B, out_channels, H, W)
    """
    def __init__(self, d_model, patch_size, out_channels, img_size):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches_h = img_size // patch_size
        self.out_channels = out_channels

        # Project from d_model to patch_area * out_channels
        patch_area = patch_size * patch_size
        self.proj = nn.Linear(d_model, patch_area * out_channels)

        # Final convolution to smooth patch boundaries
        self.smooth = nn.Conv2d(out_channels, out_channels, 3, padding=1)

    def forward(self, x):
        B, N, D = x.shape
        # Project to pixel space
        x = self.proj(x)  # (B, N, patch_area * out_channels)
        # Reshape to image grid
        pH = self.num_patches_h
        p = self.patch_size
        C = self.out_channels
        x = x.view(B, pH, pH, p, p, C)
        x = x.permute(0, 5, 1, 3, 2, 4).contiguous()  # (B, C, pH, p, pH, p)
        x = x.view(B, C, pH * p, pH * p)  # (B, C, H, W)
        # Smooth patch boundaries
        x = self.smooth(x)
        return x

---
## 5. Pure ViT for Image Restoration

The complete architecture:

```
PatchEmbed(3, 4, 256) → TransformerEncoder(L=6, H=8, d=256) → 
LayerNorm → Unpatchify(256, 4, 3, 32) → Output image
```

Note: No [CLS] token needed — we want all patch outputs, not just one.

In [ ]:
class ViTRestoration(nn.Module):
    """Vision Transformer for image restoration.
    
    No [CLS] token — all patch outputs are used for reconstruction.
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3, out_channels=3,
                 d_model=256, num_heads=8, num_layers=6, expansion=4, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.d_model = d_model

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)

        self.encoder = nn.Sequential(*[
            TransformerBlock(d_model, num_heads, expansion, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.unpatchify = Unpatchify(d_model, patch_size, out_channels, img_size)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x, return_attn=False):
        x = self.patch_embed(x)
        # Pass through transformer, optionally collecting attention weights
        attn_weights = []
        for block in self.encoder:
            if return_attn:
                x, weights = block(x, return_weights=True)
                attn_weights.append(weights)
            else:
                x = block(x)
        x = self.norm(x)
        out = self.unpatchify(x)
        if return_attn:
            return out, attn_weights
        return out


vit_restore = ViTRestoration(
    img_size=32, patch_size=4, in_channels=3, out_channels=3,
    d_model=256, num_heads=8, num_layers=6
).to(device)

total = sum(p.numel() for p in vit_restore.parameters())
print(f"ViT Restoration — Total parameters: {total:,}")

# Verify output shape
x = torch.randn(1, 3, 32, 32).to(device)
out = vit_restore(x)
print(f"Input: {x.shape} → Output: {out.shape}")

---
## 6. Hybrid CNN+ViT for Restoration

The hybrid model combines the best of both worlds:

1. **CNN encoder** extracts local features (edges, textures, patterns)
2. **ViT** applies global attention across patch features
3. **CNN decoder** reconstructs the image with learned upsampling

```
Input → CNN Encoder (local features) → Reshape to patches → 
Transformer (global attention) → Reshape to grid → CNN Decoder → Output
```

This is like having both a **local spatial filter** AND **global context** — best of both worlds.

In [ ]:
class CNNEncoder(nn.Module):
    """CNN encoder for local feature extraction."""
    def __init__(self, in_channels=3, base_channels=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 → 16
            nn.Conv2d(base_channels, base_channels*2, 3, padding=1),
            nn.BatchNorm2d(base_channels*2), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels*2, base_channels*2, 3, padding=1),
            nn.BatchNorm2d(base_channels*2), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 → 8
        )
    def forward(self, x):
        return self.net(x)

class CNNDecoder(nn.Module):
    """CNN decoder for image reconstruction."""
    def __init__(self, base_channels=64, out_channels=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(base_channels*2, base_channels, 2, stride=2),  # 8 → 16
            nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(base_channels, base_channels//2, 2, stride=2),  # 16 → 32
            nn.BatchNorm2d(base_channels//2), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels//2, out_channels, 3, padding=1),
        )
    def forward(self, x):
        return self.net(x)

class HybridRestoration(nn.Module):
    """Hybrid CNN+ViT for image restoration.
    
    CNN Encoder → Reshape to Patches → Transformer → Reshape to Grid → CNN Decoder
    """
    def __init__(self, img_size=32, in_channels=3, out_channels=3,
                 base_channels=64, d_model=128, num_heads=4, num_layers=4):
        super().__init__()

        self.cnn_encoder = CNNEncoder(in_channels, base_channels)
        # After CNN encoder: (base*2, 8, 8) → we treat each spatial position as a "patch"
        # 8*8 = 64 patches, each with base*2 features
        self.patch_proj = nn.Linear(base_channels*2, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, 64, d_model) * 0.02)
        self.dropout = nn.Dropout(0.1)

        self.transformer = nn.Sequential(*[
            TransformerBlock(d_model, num_heads, expansion=4, dropout=0.1)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.patch_unproj = nn.Linear(d_model, base_channels*2)
        self.cnn_decoder = CNNDecoder(base_channels, out_channels)

    def forward(self, x):
        # CNN Encoder: (B, 3, 32, 32) → (B, C*2, 8, 8)
        feat = self.cnn_encoder(x)
        B, C, H, W = feat.shape

        # Reshape to sequence: (B, H*W, C)
        seq = feat.flatten(2).transpose(1, 2)

        # Project to transformer dim + add position encoding
        seq = self.patch_proj(seq)
        seq = seq + self.pos_embed
        seq = self.dropout(seq)

        # Transformer
        seq = self.transformer(seq)
        seq = self.norm(seq)

        # Project back to CNN feature dim
        seq = self.patch_unproj(seq)

        # Reshape back to grid: (B, H*W, C) → (B, C, H, W)
        feat_out = seq.transpose(1, 2).view(B, C, H, W)

        # CNN Decoder: (B, C*2, 8, 8) → (B, 3, 32, 32)
        out = self.cnn_decoder(feat_out)
        return out


hybrid = HybridRestoration(
    img_size=32, in_channels=3, out_channels=3,
    base_channels=64, d_model=128, num_heads=4, num_layers=4
).to(device)

total = sum(p.numel() for p in hybrid.parameters())
print(f"Hybrid CNN+ViT — Total parameters: {total:,}")

x = torch.randn(1, 3, 32, 32).to(device)
out = hybrid(x)
print(f"Input: {x.shape} → Output: {out.shape}")

---
## 7. CNN U-Net Baseline (from Notebook 2)

We reuse the U-Net from Notebook 2 as a baseline for comparison. This is the same architecture — no changes needed.

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, base_channels=64):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 3, padding=1), nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels, 3, padding=1), nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
        )
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(
            nn.Conv2d(base_channels, base_channels*2, 3, padding=1), nn.BatchNorm2d(base_channels*2), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels*2, base_channels*2, 3, padding=1), nn.BatchNorm2d(base_channels*2), nn.ReLU(inplace=True),
        )
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = nn.Sequential(
            nn.Conv2d(base_channels*2, base_channels*4, 3, padding=1), nn.BatchNorm2d(base_channels*4), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels*4, base_channels*4, 3, padding=1), nn.BatchNorm2d(base_channels*4), nn.ReLU(inplace=True),
        )
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(base_channels*4, base_channels*8, 3, padding=1), nn.BatchNorm2d(base_channels*8), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels*8, base_channels*4, 3, padding=1), nn.BatchNorm2d(base_channels*4), nn.ReLU(inplace=True),
        )
        self.up3 = nn.ConvTranspose2d(base_channels*4, base_channels*4, 2, stride=2)
        self.dec3 = nn.Sequential(
            nn.Conv2d(base_channels*8, base_channels*2, 3, padding=1), nn.BatchNorm2d(base_channels*2), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels*2, base_channels*2, 3, padding=1), nn.BatchNorm2d(base_channels*2), nn.ReLU(inplace=True),
        )
        self.up2 = nn.ConvTranspose2d(base_channels*2, base_channels*2, 2, stride=2)
        self.dec2 = nn.Sequential(
            nn.Conv2d(base_channels*4, base_channels, 3, padding=1), nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels, base_channels, 3, padding=1), nn.BatchNorm2d(base_channels), nn.ReLU(inplace=True),
        )
        self.up1 = nn.ConvTranspose2d(base_channels, base_channels, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(base_channels*2, base_channels//2, 3, padding=1), nn.BatchNorm2d(base_channels//2), nn.ReLU(inplace=True),
            nn.Conv2d(base_channels//2, out_channels, 3, padding=1),
        )

    def forward(self, x):
        s1 = self.enc1(x)
        p1 = self.pool1(s1)
        s2 = self.enc2(p1)
        p2 = self.pool2(s2)
        s3 = self.enc3(p2)
        p3 = self.pool3(s3)
        b = self.bottleneck(p3)
        d3 = self.up3(b)
        d3 = torch.cat([d3, s3], dim=1)
        d3 = self.dec3(d3)
        d2 = self.up2(d3)
        d2 = torch.cat([d2, s2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        d1 = torch.cat([d1, s1], dim=1)
        out = self.dec1(d1)
        return out

unet = UNet(base_channels=64).to(device)
total = sum(p.numel() for p in unet.parameters())
print(f"U-Net — Total parameters: {total:,}")

---
## 8. Training Infrastructure

Training and evaluation functions — same pattern as Notebooks 1 and 2.

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    total = 0
    for degraded, clean in loader:
        degraded, clean = degraded.to(device), clean.to(device)
        output = model(degraded)
        loss = loss_fn(output, clean)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * degraded.size(0)
        total += degraded.size(0)
    return total_loss / total

@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, total_psnr, total_ssim, total = 0, 0, 0, 0
    for degraded, clean in loader:
        degraded, clean = degraded.to(device), clean.to(device)
        output = model(degraded)
        loss = loss_fn(output, clean)
        total_loss += loss.item() * degraded.size(0)
        total_psnr += compute_psnr(output, clean) * degraded.size(0)
        total_ssim += compute_ssim(output, clean) * degraded.size(0)
        total += degraded.size(0)
    return total_loss / total, total_psnr / total, total_ssim / total

def train_model(model, train_loader, val_loader, name, epochs=50, lr=1e-3, warmup=5, use_warmup=False):
    loss_fn = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    if use_warmup:
        def lr_lambda(epoch):
            if epoch < warmup: return epoch / warmup
            progress = (epoch - warmup) / (epochs - warmup)
            return 0.5 * (1 + math.cos(math.pi * progress))
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    else:
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    history = {"train_loss": [], "val_loss": [], "psnr": [], "ssim": []}
    best_psnr = 0
    print(f"\n{'='*50}\nTraining {name}\n{'='*50}")
    print(f"{'Ep':>3} | {'TrLoss':>8} | {'VLoss':>8} | {'PSNR':>6} | {'SSIM':>6} | {'Time':>5}")
    print("-" * 50)

    for epoch in range(epochs):
        t0 = time.time()
        tl = train_epoch(model, train_loader, loss_fn, optimizer, device)
        vl, psnr, ssim = evaluate(model, val_loader, loss_fn, device)
        scheduler.step()
        elapsed = time.time() - t0

        history["train_loss"].append(tl)
        history["val_loss"].append(vl)
        history["psnr"].append(psnr)
        history["ssim"].append(ssim)

        if psnr > best_psnr:
            best_psnr = psnr
            torch.save(model.state_dict(), f"best_{name.lower().replace(' ', '_')}.pt")

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"{epoch+1:3d} | {tl:8.4f} | {vl:8.4f} | {psnr:5.2f} | {ssim:5.4f} | {elapsed:4.1f}s")

    model.load_state_dict(torch.load(f"best_{name.lower().replace(' ', '_')}.pt", map_location=device, weights_only=True))
    print(f"✅ {name} — Best val PSNR: {best_psnr:.2f} dB")
    return history

---
## 9. Train All Models on Denoising

We train all three models on the same denoising task (σ=0.2) and compare results.

**Training config:**
- U-Net: 30 epochs, lr=1e-3, CosineAnnealingLR
- ViT: 50 epochs, lr=3e-4, warmup + cosine decay (ViT needs more epochs and lower LR)
- Hybrid: 50 epochs, lr=1e-3, CosineAnnealingLR

In [ ]:
# Train U-Net on denoising (30 epochs — smaller model, faster convergence)
unet_dn = UNet(base_channels=64).to(device)
history_unet_dn = train_model(unet_dn, train_loader_dn, val_loader_dn, "U-Net Denoiser", epochs=30, lr=1e-3)

In [ ]:
# Train ViT on denoising (50 epochs — needs more)
vit_dn = ViTRestoration(
    img_size=32, patch_size=4, in_channels=3, out_channels=3,
    d_model=256, num_heads=8, num_layers=6
).to(device)
history_vit_dn = train_model(vit_dn, train_loader_dn, val_loader_dn,
                              "ViT Denoiser", epochs=50, lr=3e-4, warmup=5, use_warmup=True)

In [ ]:
# Train Hybrid on denoising (50 epochs)
hybrid_dn = HybridRestoration(
    img_size=32, in_channels=3, out_channels=3,
    base_channels=64, d_model=128, num_heads=4, num_layers=4
).to(device)
history_hybrid_dn = train_model(hybrid_dn, train_loader_dn, val_loader_dn,
                                 "Hybrid Denoiser", epochs=50, lr=1e-3)

---
## 10. Compare All Three Models

Let's see which architecture performs best on denoising.

In [ ]:
# Comparison table
models_info = [
    ("U-Net", history_unet_dn, sum(p.numel() for p in unet_dn.parameters())),
    ("ViT", history_vit_dn, sum(p.numel() for p in vit_dn.parameters())),
    ("Hybrid", history_hybrid_dn, sum(p.numel() for p in hybrid_dn.parameters())),
]

print(f"{'Model':<10} {'Params':>10} {'PSNR (dB)':>10} {'SSIM':>8}")
print("-" * 40)
for name, hist, params in models_info:
    print(f"{name:<10} {params:>10,} {hist['psnr'][-1]:>8.2f} {hist['ssim'][-1]:>8.4f}")

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, hist, _ in models_info:
    axes[0].plot(hist["train_loss"], label=name)
    axes[1].plot(hist["psnr"], label=name)
    axes[2].plot(hist["ssim"], label=name)

axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("PSNR (dB)"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].set_title("SSIM"); axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.suptitle("Denoising: CNN U-Net vs ViT vs Hybrid", fontsize=14)
plt.tight_layout()
plt.show()

### Visual Comparison

Let's see the actual reconstructions side by side.

In [ ]:
unet_dn.eval(); vit_dn.eval(); hybrid_dn.eval()
degraded, clean = next(iter(test_loader_dn))
degraded, clean = degraded.to(device), clean.to(device)

with torch.no_grad():
    out_unet = unet_dn(degraded)
    out_vit = vit_dn(degraded)
    out_hybrid = hybrid_dn(degraded)

fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for i in range(5):
    def show(img_tensor, ax):
        img = (img_tensor * torch.tensor(CIFAR_STD).view(3,1,1).to(img_tensor.device)
               + torch.tensor(CIFAR_MEAN).view(3,1,1).to(img_tensor.device)).clamp(0,1)
        ax.imshow(img.cpu().permute(1,2,0)); ax.axis("off")

    show(degraded[i], axes[0, i]); axes[0, i].set_title("Degraded")
    show(out_unet[i], axes[1, i]); axes[1, i].set_title("U-Net")
    show(out_vit[i], axes[2, i]); axes[2, i].set_title("ViT")
    show(out_hybrid[i], axes[3, i]); axes[3, i].set_title("Hybrid")
    show(clean[i], axes[4, i]); axes[4, i].set_title("Ground Truth")

plt.suptitle("Denoising: Model Comparison", fontsize=14)
plt.tight_layout()
plt.show()

---
## 11. Deblurring Comparison

Now let's test all three models on the harder task: deblurring (inverting a Gaussian PSF).

**The question:** Does global attention help more for deblurring than for denoising?

Hypothesis: Yes — because blur destroys *structured* information (edges, textures) that require understanding of the whole image to reconstruct. ViT's global attention can look at sharp edges far away to infer what the blur kernel did.

In [ ]:
# Train all three on deblurring
unet_bl = UNet(base_channels=64).to(device)
history_unet_bl = train_model(unet_bl, train_loader_bl, val_loader_bl, "U-Net Deblurrer", epochs=30, lr=1e-3)

vit_bl = ViTRestoration(img_size=32, patch_size=4, in_channels=3, out_channels=3,
                         d_model=256, num_heads=8, num_layers=6).to(device)
history_vit_bl = train_model(vit_bl, train_loader_bl, val_loader_bl,
                              "ViT Deblurrer", epochs=50, lr=3e-4, warmup=5, use_warmup=True)

hybrid_bl = HybridRestoration(img_size=32, in_channels=3, out_channels=3,
                               base_channels=64, d_model=128, num_heads=4, num_layers=4).to(device)
history_hybrid_bl = train_model(hybrid_bl, train_loader_bl, val_loader_bl,
                                 "Hybrid Deblurrer", epochs=50, lr=1e-3)

In [ ]:
# Comparison table for deblurring
print(f"{'Model':<10} {'Params':>10} {'PSNR (dB)':>10} {'SSIM':>8}")
print("-" * 40)
deblur_models = [
    ("U-Net", history_unet_bl, sum(p.numel() for p in unet_bl.parameters())),
    ("ViT", history_vit_bl, sum(p.numel() for p in vit_bl.parameters())),
    ("Hybrid", history_hybrid_bl, sum(p.numel() for p in hybrid_bl.parameters())),
]
for name, hist, params in deblur_models:
    print(f"{name:<10} {params:>10,} {hist['psnr'][-1]:>8.2f} {hist['ssim'][-1]:>8.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, hist, _ in deblur_models:
    axes[0].plot(hist["train_loss"], label=name)
    axes[1].plot(hist["psnr"], label=name)
    axes[2].plot(hist["ssim"], label=name)
axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title("PSNR (dB)"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].set_title("SSIM"); axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.suptitle("Deblurring: CNN U-Net vs ViT vs Hybrid", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
unet_bl.eval(); vit_bl.eval(); hybrid_bl.eval()
degraded, clean = next(iter(test_loader_bl))
degraded, clean = degraded.to(device), clean.to(device)

with torch.no_grad():
    out_unet = unet_bl(degraded)
    out_vit = vit_bl(degraded)
    out_hybrid = hybrid_bl(degraded)

fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for i in range(5):
    def show(img_tensor, ax):
        img = (img_tensor * torch.tensor(CIFAR_STD).view(3,1,1).to(img_tensor.device)
               + torch.tensor(CIFAR_MEAN).view(3,1,1).to(img_tensor.device)).clamp(0,1)
        ax.imshow(img.cpu().permute(1,2,0)); ax.axis("off")
    show(degraded[i], axes[0, i]); axes[0, i].set_title("Blurred")
    show(out_unet[i], axes[1, i]); axes[1, i].set_title("U-Net")
    show(out_vit[i], axes[2, i]); axes[2, i].set_title("ViT")
    show(out_hybrid[i], axes[3, i]); axes[3, i].set_title("Hybrid")
    show(clean[i], axes[4, i]); axes[4, i].set_title("Ground Truth")
plt.suptitle("Deblurring: Model Comparison", fontsize=14)
plt.tight_layout()
plt.show()

---
## 12. Attention Map Visualization for Restoration

Let's visualize what the ViT attends to when reconstructing a degraded image. We use the same Attention Rollout technique from Day 26.

For restoration, attention maps show which patches the model "looks at" when reconstructing each patch.

In [ ]:
@torch.no_grad()
def attention_rollout(vit_model, img, device):
    """Compute attention rollout for restoration ViT."""
    vit_model.eval()
    img = img.unsqueeze(0).to(device)
    
    # Forward pass with attention capture
    _, attn_weights = vit_model(img, return_attn=True)
    
    # attn_weights: list of (B, H, N, N) for each layer
    # Average over heads
    rollout = torch.eye(attn_weights[0].shape[-1], device=device)
    for attn in attn_weights:
        avg_attn = attn.mean(dim=1)  # (B, N, N)
        rollout = rollout @ avg_attn[0]
    
    rollout = rollout[1:, 1:]  # Remove [CLS] — we didn't use it, but the attention still has it
    return rollout.cpu()


# Visualize attention for a degraded image
degraded_sample, clean_sample = test_dn[0]

rollout = attention_rollout(vit_dn, degraded_sample, device)
N = rollout.shape[0]  # Number of patches (64 for 4×4 patches on 32×32)
side = int(math.sqrt(N))  # 8

# Reshape rollout to grid format: for each patch, show which patches it attends to
# We'll show the average attention each patch receives from all other patches
avg_attention = rollout.mean(dim=0).reshape(side, side)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Original images
def denorm(img_t):
    return (img_t * torch.tensor(CIFAR_STD).view(3,1,1) + torch.tensor(CIFAR_MEAN).view(3,1,1)).clamp(0,1).permute(1,2,0)

axes[0].imshow(denorm(degraded_sample))
axes[0].set_title("Degraded Input"); axes[0].axis("off")

axes[1].imshow(denorm(clean_sample))
axes[1].set_title("Ground Truth"); axes[1].axis("off")

# Attention heatmap overlaid on patches
im = axes[2].imshow(avg_attention, cmap="hot", interpolation="nearest")
axes[2].set_title("Average Attention per Patch")
axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.suptitle("ViT Attention Map for Denoising", fontsize=14)
plt.tight_layout()
plt.show()
print("📝 Hotter = patch receives more attention from other patches.")
print("    This shows which spatial regions the model considers important for reconstruction.")

---
## 📊 Final Comparison Table

After training all models, we can summarize:

In [ ]:
print("=" * 70)
print(f"{'Model':<15} {'Task':<12} {'Params':>10} {'PSNR':>8} {'SSIM':>8}")
print("=" * 70)

all_results = [
    ("U-Net",        "Denoising",  sum(p.numel() for p in unet_dn.parameters()), history_unet_dn["psnr"][-1], history_unet_dn["ssim"][-1]),
    ("ViT",          "Denoising",  sum(p.numel() for p in vit_dn.parameters()), history_vit_dn["psnr"][-1], history_vit_dn["ssim"][-1]),
    ("Hybrid",       "Denoising",  sum(p.numel() for p in hybrid_dn.parameters()), history_hybrid_dn["psnr"][-1], history_hybrid_dn["ssim"][-1]),
    ("U-Net",        "Deblurring", sum(p.numel() for p in unet_bl.parameters()), history_unet_bl["psnr"][-1], history_unet_bl["ssim"][-1]),
    ("ViT",          "Deblurring", sum(p.numel() for p in vit_bl.parameters()), history_vit_bl["psnr"][-1], history_vit_bl["ssim"][-1]),
    ("Hybrid",       "Deblurring", sum(p.numel() for p in hybrid_bl.parameters()), history_hybrid_bl["psnr"][-1], history_hybrid_bl["ssim"][-1]),
]

for name, task, params, psnr, ssim in all_results:
    print(f"{name:<15} {task:<12} {params:>10,} {psnr:>7.2f} dB {ssim:>7.4f}")
print("=" * 70)

---
## 🧪 Exercises

### Exercise 1: Patch Size
Vary patch size in the ViT (4×4, 8×8, 16×16). How does it affect restoration quality and training speed?

### Exercise 2: Attention Comparison
Compare attention maps for denoising vs. deblurring. Do the attention patterns look different?

### Exercise 3: Transformer Depth
Train ViT with different number of layers (L=2, 4, 6, 8). Do you see diminishing returns?

### Exercise 4: Real-World Image
Load your own photo, add synthetic noise/blur, and run all three models. Which does best?

In [ ]:
# Exercise 1: Your code here

In [ ]:
# Exercise 2: Your code here

In [ ]:
# Exercise 3: Your code here

In [ ]:
# Exercise 4: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **ViT for restoration** | Same backbone as classification, but Unpatchify head instead of [CLS] + MLP |
| **Global attention** | ViT sees all patches — helps for structured degradation (blur) more than random noise |
| **Hybrid CNN+ViT** | CNN for local features, ViT for global context, CNN decoder for reconstruction |
| **CNN vs ViT trade-off** | CNN: fewer params, faster, good for local noise. ViT: global context, more params, needs more data/epochs |
| **Hybrid wins** | Usually the best practical choice — combines locality and globality |
| **Attention maps** | Show which patches the model "looks at" — interpretable restoration |

### What You've Learned Across All 3 Extension Notebooks

1. **Any classifier can become an image-to-image model** — change the head and loss (Notebook 1)
2. **U-Net with skip connections** preserves fine details for restoration (Notebook 2)
3. **PSNR (dB) and SSIM** quantify restoration quality — familiar from camera/optics work (Notebook 2)
4. **ViT can do restoration too** — same backbone, different head (Notebook 3)
5. **Hybrid CNN+ViT** combines local features and global context for best results (Notebook 3)
6. **Synthetic degradation** gives full control over noise type and level for reproducible research